# LTLTRA001 | Tracey Letlape | CSC3042F

## Multi-Layer Perceptron (MLP) vs Convolutional Neural Network (CNN)

Purpose of the assignment is to investigate the MLP and CNN architectures throught the following complementary lenses:
* Complexity Ladder: When MLP begin to degrade and why CNNs maintain performane
* Parameter Budget: Fix computational budget, and determine how much architecture design matters for each model class.

In [14]:
# Necessary imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset
from itertools import product
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import pandas as pd
import random
import logging
import os
from datetime import datetime

# DATA LOADING

* Define a transformer for each dataset
* Get both the training and the test datasets
* Split the training dataset into the training and validation datasets

In [2]:
# Define a reusable helper method to split a dataset
def _split_dataset(dataset: datasets.MNIST | datasets.CIFAR10, train_ratio: float = 0.8, seed: int = 42) -> tuple[Subset, Subset]:
    # Compute split sizes
    total_size = len(dataset)

    train_size = int(train_ratio * total_size)
    val_size = total_size - train_size

    # Reproducible split
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size]
    )

    return train_dataset, val_dataset

In [3]:
# Define MNIST normalisation parameters
MNIST_MEAN = (0.1307,)
MNIST_STD = (0.30811,)

# Define a MNIST transfom
_mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

# Get the MNIST datasets
_mnist_full_train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=_mnist_transform
)

MNIST_TEST_DATASET = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=_mnist_transform
)

# Split the training dataset into train and val datasets
MNIST_TRAIN_DATASET, MNIST_VAL_DATASET = _split_dataset(_mnist_full_train_dataset)

print(f"Train size: {len(MNIST_TRAIN_DATASET)}")
print(f"validation size: {len(MNIST_VAL_DATASET)}")
print(f"Test size: {len(MNIST_TEST_DATASET)}")

Train size: 48000
validation size: 12000
Test size: 10000


In [4]:
# Define CIFAR Normalisation parameters
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

# Define a CIFAR train transform
_cifar_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# Define a CIFAR test transform
_cifar_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

_cifar_full_train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=_cifar_train_transform
)

CIFAR_TEST_DATASET = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=_cifar_test_transform,
)

# Split the training dataset into train and val datasets
CIFAR_TRAIN_DATASET, CIFAR_VAL_DATASET = _split_dataset(_cifar_full_train_dataset)

print(f"Train size: {len(CIFAR_TRAIN_DATASET)}")
print(f"validation size: {len(CIFAR_VAL_DATASET)}")
print(f"Test size: {len(CIFAR_TEST_DATASET)}")

Train size: 40000
validation size: 10000
Test size: 10000


# PART A

* Implementation: Code for both architectures and training loop
* Results table: Test Accuracy vs Dataset combinations
* Training Curves: Loss and accuracy plots on a single figure
* Analysis


In [15]:
# Reproducibility
seed = random.seed(42)
torch.manual_seed(42)
np.random.seed(42)

# GLOBAL VARIABLES
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = 10
MNIST_CLASSES = ('0', '1', '2', '3', '4', '5', '6', '7', '8', '9') 
CIFAR_CLASSES = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

Using device: cpu
PyTorch version: 2.8.0+cu129


## Multi-Layer Perceptron
* Input Layer, 3 Hidden Layers, Output Layer
* Architecture: [input_dim -> 256 -> 128 -> 64 -> 32 -> 10]
* Uses the ReLU activation function for all layers except the output layer which uses sigmoid.

In [6]:
# Define a fixed MLP
class MLP(nn.Module):
    """
        Feedforward network implementing MLP.
        Architecture: [input_dim -> *hidden_dims -> output_dim]
    """
    def __init__(self, input_dim, num_classes: int = 10):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, 256)
        self.h1 = nn.Linear(256, 128)
        self.h2 = nn.Linear(128, 64)
        self.h3 = nn.Linear(64, 32)
        self.output_layer = nn.Linear(32, num_classes)

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        # Input layer
        self.z1 = self.input_layer(x)
        self.a1 = F.relu(self.z1)

        # First hidden layer
        self.z2 = self.h1(self.a1)
        self.a2 = F.relu(self.z2)

        # Second hidden layer
        self.z3 = self.h2(self.a2)
        self.a3 = F.relu(self.z3)

        # Third hidden layer
        self.z4 = self.h3(self.a3)
        self.a4 = F.relu(self.z4)

        # Output layer
        self.z5 = self.output_layer(self.a4)
        self.a5 = F.sigmoid(self.z4)

        return self.a5

## Convolutional Neural Network (CNNs)
* Defined a reusable ConvBlock class and the CNN model class
* Input Layer, 3 Hidden Layers, Output Layer
* Architecture: [input_dim -> 256 -> 128 -> 64 -> 32 -> 10]
* Uses the ReLU activation function for all layers except the output layer which uses sigmoid.

In [7]:
class ConvBlock(nn.Module):
    """
    Conv2d -> Batchnorm -> ReLU
    """

    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel,
                      stride=stride, padding=padding, bias=False
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

In [8]:
class CNN(nn.Module):
    def __init__(self, input_channels, dropout=0.5, num_classes=10):
        super().__init__()

        # ── Stage 1: extract low-level features ───────────────────────────
        self.stage1 = nn.Sequential(
            ConvBlock(input_channels,  32),      # Transformation into desired shape
            ConvBlock(32, 32),      # Reform
            nn.MaxPool2d(2, 2),             # HxW -> H/2 x W/2
        )

        # ── Stage 2: higher-level features ────────────────────────────────
        self.stage2 = nn.Sequential(
            ConvBlock(32, 64),
            ConvBlock(64, 64),
            nn.MaxPool2d(2, 2),             # H/2 x W/2 -> H/4 x W/4
        )

        # ── Stage 3: abstract representations ─────────────────────────────
        self.stage3 = nn.Sequential(
            ConvBlock(64, 128),
        )

        # ── Classifier head ───────────────────────────────────────────────
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),       # 128xH'xW' -> 128x1x1
            nn.Flatten(),                       #  -> 128
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.head(x)
        return x

## Training Loop

In [ ]:
def train_model(model: MLP | CNN, loader: DataLoader, criterion: nn.Module, optimiser: optim.Optimizer, device: torch.device | str = 'cpu'):
    # Move model to device
    model.to(device)
    
    # Set model to training mode
    model.train()

    total_loss, total_correct, total_samples = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        # Reset gradients to zero
        optimiser.zero_grad()

        # Forward pass & loss computation
        logits = model(images)
        loss = criterion(logits, labels)

        # Backward pass & parameter update
        loss.backward()
        optimiser.step()

        # Update metrics
        batch_size = images.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_samples += batch_size

    # Compute avarage metrics
    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples

    return avg_loss, avg_acc

@torch.no_grad()
def evaluate_model(model: MLP | CNN, loader: DataLoader, criterion: nn.Module, device: torch.device | str = 'cpu'):
    # Move model to device
    model.to(device)
    
    # Set the model to evaluation mode
    model.eval()

    total_loss, total_correct, total_samples = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        logits = model(images)
        loss = criterion(logits, labels)

        # Update metrics
        batch_size = images.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_samples += batch_size

    # Compute avarage metrics
    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples

    return avg_loss, avg_acc


def training_loop(model_name: str, train_dataset: Subset, val_dataset: Subset, lr: float = 1e-3, batch_size: int = 128, dropout: float = 0.5, device: torch.device = DEVICE):
    # Define the number of epochs
    num_epochs = 10

    # Define the patience limit to stop when there is no improvement
    PATIENCE = 5

    # Define the metrics to track performance
    best_val_loss = float("inf")
    epochs_no_improve = 0

    # Use pin_memory only if using CUDA
    pin_memory = device == 'cuda' or str(device).startswith("cuda")
    
    # Define loader
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=pin_memory)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=pin_memory)
    
    # Get input_size
    temp, _ = next(iter(train_loader))

    # Define the model
    model = None
    if model_name == 'MLP':
        input_size = temp[0].numel()
        model = MLP(input_size, NUM_CLASSES)

    elif model_name == 'CNN':
        input_channels = temp.shape[1]
        model= CNN(input_channels=input_channels, dropout=dropout, num_classes=NUM_CLASSES)

    else:
        raise ValueError(f"model_name must be either 'MLP' or 'CNN'")
    
    # Move model to device
    model = model.to(device)
    
    # Use CrossEntropyLoss as the loss function
    criterion = nn.CrossEntropyLoss()

    # Use AdamW with weight_decay as 1e-4 as the optimiser
    optimiser = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # Define a scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=num_epochs)

    # Store metrics
    history = {
        'epoch': [],
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': [],
        'lr': [],
        'batch_size': [],
        'dropout': []
    }

    # Run through epochs
    for epoch in range(1, num_epochs+1):
        tr_loss, tr_acc = train_model(model, train_loader, criterion, optimiser, device)
        va_loss, va_acc = evaluate_model(model, val_loader, criterion, device)
        
        current_lr = scheduler.get_last_lr()[0]

        history['epoch'].append(epoch)
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(va_acc)
        history['lr'].append(current_lr)
        history['batch_size'].append(batch_size)
        history['dropout'].append(dropout)

        scheduler.step()

        if va_loss < best_val_loss:
            best_val_loss = va_loss
            epochs_no_improve = 0   # Reset state
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break
        
    return model, history

#### MLP Training

In [10]:
# Train the MLP model on MNIST data using default parameters
mnist_mlp_model, mnist_mlp_history = training_loop('MLP', MNIST_TRAIN_DATASET, MNIST_VAL_DATASET)

# Default history
defualt_mnist_mlp_history = {
    'epoch': mnist_mlp_history['epoch'],
    'train_loss': mnist_mlp_history['train_loss'],
    'val_loss': mnist_mlp_history['val_loss'],
    'train_acc': mnist_mlp_history['train_acc'],
    'val_acc': mnist_mlp_history['val_acc'],
    'lr': mnist_mlp_history['lr']
}

mnist_mlp_history_df = pd.DataFrame(defualt_mnist_mlp_history)
print(mnist_mlp_history_df)

   epoch  train_loss  val_loss  train_acc   val_acc        lr
0      1    2.907730  2.895545   0.097146  0.104333  0.001000
1      2    2.895545  2.895545   0.097312  0.104333  0.000976
2      3    2.895545  2.895545   0.097312  0.104333  0.000905
3      4    2.895545  2.895545   0.097312  0.104333  0.000794
4      5    2.895545  2.895545   0.097312  0.104333  0.000655
5      6    2.895545  2.895545   0.097312  0.104333  0.000500
6      7    2.895545  2.895545   0.097312  0.104333  0.000345
7      8    2.895545  2.895545   0.097312  0.104333  0.000206
8      9    2.895545  2.895545   0.097312  0.104333  0.000095
9     10    2.895545  2.895545   0.097312  0.104333  0.000024


In [11]:
# Train the MLP model on CIFAR data using default parameters
cifar_mlp_model, cifar_mlp_history = training_loop('MLP', CIFAR_TRAIN_DATASET, CIFAR_VAL_DATASET)

# Default history
defualt_cifar_mlp_history = {
    'epoch': cifar_mlp_history['epoch'],
    'train_loss': cifar_mlp_history['train_loss'],
    'val_loss': cifar_mlp_history['val_loss'],
    'train_acc': cifar_mlp_history['train_acc'],
    'val_acc': cifar_mlp_history['val_acc'],
    'lr': cifar_mlp_history['lr']
}

cifar_mlp_history_df = pd.DataFrame(defualt_cifar_mlp_history)
print(cifar_mlp_history_df)

   epoch  train_loss  val_loss  train_acc  val_acc        lr
0      1    2.903721  2.881849   0.117525   0.1296  0.001000
1      2    2.871748  2.873319   0.133975   0.1159  0.000976
2      3    2.873002  2.870723   0.125625   0.1228  0.000905
3      4    2.869252  2.872929   0.117625   0.1173  0.000794
4      5    2.868285  2.869725   0.122325   0.1215  0.000655
5      6    2.866858  2.870452   0.117150   0.1170  0.000500
6      7    2.865473  2.866865   0.116900   0.1164  0.000345
7      8    2.864636  2.865519   0.115425   0.1179  0.000206
8      9    2.863602  2.865025   0.116575   0.1153  0.000095
9     10    2.863137  2.865889   0.115025   0.1152  0.000024


#### CNN Training

In [12]:
# Train the CNN model on MNIST data using default parameters
mnist_cnn_model, mnist_cnn_history = training_loop('CNN', MNIST_TRAIN_DATASET, MNIST_VAL_DATASET)

# Default history
defualt_mnist_cnn_history = {
    'epoch': mnist_cnn_history['epoch'],
    'train_loss': mnist_cnn_history['train_loss'],
    'val_loss': mnist_cnn_history['val_loss'],
    'train_acc': mnist_cnn_history['train_acc'],
    'val_acc': mnist_cnn_history['val_acc'],
    'lr': mnist_cnn_history['lr']
}

mnist_cnn_history_df = pd.DataFrame(defualt_mnist_cnn_history)
print(mnist_cnn_history_df)

   epoch  train_loss  val_loss  train_acc   val_acc        lr
0      1    0.368836  0.138304   0.920146  0.962083  0.001000
1      2    0.076354  0.045704   0.981229  0.986583  0.000976
2      3    0.053986  0.053114   0.985479  0.984000  0.000905
3      4    0.041622  0.036014   0.988292  0.989000  0.000794
4      5    0.033880  0.038089   0.990500  0.987667  0.000655
5      6    0.025893  0.025246   0.992667  0.992500  0.000500
6      7    0.019840  0.025294   0.994729  0.991750  0.000345
7      8    0.014625  0.022280   0.996208  0.993583  0.000206
8      9    0.012326  0.019363   0.997146  0.994083  0.000095
9     10    0.010231  0.019199   0.997708  0.994083  0.000024


In [13]:
# Train the MLP model on CIFAR data using default parameters
cifar_cnn_model, cifar_cnn_history = training_loop('CNN', CIFAR_TRAIN_DATASET, CIFAR_VAL_DATASET)

# Default history
defualt_cifar_cnn_history = {
    'epoch': cifar_cnn_history['epoch'],
    'train_loss': cifar_cnn_history['train_loss'],
    'val_loss': cifar_cnn_history['val_loss'],
    'train_acc': cifar_cnn_history['train_acc'],
    'val_acc': cifar_cnn_history['val_acc'],
    'lr': cifar_cnn_history['lr']
}

cifar_cnn_history_df = pd.DataFrame(defualt_cifar_cnn_history)
print(cifar_cnn_history_df)

   epoch  train_loss  val_loss  train_acc  val_acc        lr
0      1    1.569255  1.350675   0.421375   0.5217  0.001000
1      2    1.219976  1.162712   0.563775   0.5800  0.000976
2      3    1.087034  1.012968   0.611550   0.6414  0.000905
3      4    0.997307  1.054622   0.644400   0.6176  0.000794
4      5    0.930825  0.861259   0.672075   0.6896  0.000655
5      6    0.873193  0.823680   0.693375   0.7091  0.000500
6      7    0.822529  0.748477   0.709500   0.7347  0.000345
7      8    0.784567  0.725499   0.727000   0.7422  0.000206
8      9    0.761581  0.714971   0.733175   0.7477  0.000095
9     10    0.744379  0.679654   0.738925   0.7540  0.000024


### TUNE HYPERPARAMETERS
* Tune the learning rates, dropouts, batch_size

In [ ]:
def _run_single_experiment(args):
    model_name, train_dataset, val_dataset, lr, batch_size, dropout, gpu_id = args
    
    if torch.cuda.is_available():
        device = torch.device(f"cuda:{gpu_id % torch.cuda.device_count()}")
    else:
        device = torch.device("cpu")

    try:
        _, history = training_loop(model_name, train_dataset, val_dataset, lr, batch_size, dropout, device)
        best_val_acc = max(history['val_acc'])
        best_epoch_index = history['val_acc'].index(best_val_acc)

        return {
            "best_train_loss": history["train_loss"][best_epoch_index],
            "best_val_loss": history["val_loss"][best_epoch_index],
            "best_train_acc": history["train_acc"][best_epoch_index],
            "best_val_acc": best_val_acc,
            "lr": lr,
            "batch_size": batch_size,
            "dropout": dropout
        }
        
    except Exception as e:
        return {
            "lr": lr,
            "batch_size": batch_size,
            "dropout": dropout,
            "error": str(e)
        }

def _setup_logger() -> logging.Logger:
    """Creates the logs directory and configures a file logger."""
    os.makedirs("logs", exist_ok=True)

    timestamp = datetime().now().strftime("%Y-%m-%d_%H-%M-%S")
    log_path = os.path.join("logs", f"tuning_{timestamp}.log")

    logger = logging.getLogger("tuning")
    logger.setLevel(logging.DEBUG)


           
def _tune_model_parallel(model_name: str, train_dataset: Subset, val_dataset: Subset, lrs: list[float], batch_sizes: list[int], dropouts: list[float], device: torch.device):
    # Define configurations
    all_configs = list(product(lrs, batch_sizes, dropouts))

    # Randomly sample 10 configs
    configs = random.sample(all_configs, min(10, len(all_configs)))

    # Determine worker cont: 1 worker per GPU, or 1 for CPU
    n_gpus = torch.cuda.device_count()
    workers = max(n_gpus, 1)

    # Package args as plain tuples (pickable)
    tasks = [
        (model_name, train_dataset, val_dataset, lr, bs, do, gpu_id)
        for gpu_id, (lr, bs, do) in enumerate(configs)
    ]

    results = []

    if workers == 1:
        for i, task in enumerate(tasks, 1):
            result = _run_single_experiment(task)
            results.append(result)
            _log_result(result, i , len(tasks))
    
    else:
        with ThreadPoolExecutor(max_workers=2) as executor:
            futures = []
            for lr, batch_size, dropout in configs:
                future = executor.submit(
                    _run_single_experiment,
                    model_name,
                    train_dataset,
                    val_dataset,
                    lr,
                    batch_size,
                    dropout,
                    device
                )
                futures.append(future)

            for i, future in enumerate(as_completed(futures), start=1):
                result = future.result()
                results.append(result)
                
                if "error" in result:
                    print(f"[{i}/{len(configs)}] Failed: {result['error']}")
                    return
        
    results_df = pd.DataFrame(results)
    return results_df.sort_values(by='best_val_acc', ascending=False).reset_index(drop=True)


In [19]:
# Tune the MNIST MLP model
lrs = [1e-3, 3e-4, 1e-4]
dropouts = [0.5, 0.3, 0.1]
batch_sizes = [64, 128, 256]

mnist_mlp_results_df = _tune_model_parallel('MLP', MNIST_TRAIN_DATASET, MNIST_VAL_DATASET, lrs, batch_sizes, dropouts, DEVICE)

print(mnist_mlp_results_df.head(10))

   best_train_loss  best_val_loss  best_train_acc  best_val_acc      lr  \
0         2.523826       2.534585        0.986938      0.975167  0.0010   
1         2.523913       2.533905        0.986000      0.973833  0.0010   
2         2.523783       2.534257        0.986208      0.973750  0.0010   
3         2.525500       2.535580        0.980208      0.970250  0.0010   
4         2.526300       2.536432        0.979667      0.969583  0.0010   
5         2.531047       2.539491        0.971083      0.964250  0.0010   
6         2.530624       2.539389        0.970542      0.962167  0.0010   
7         2.533633       2.540577        0.966000      0.961000  0.0003   
8         2.535832       2.541023        0.963187      0.958750  0.0003   
9         2.533567       2.541572        0.965250      0.958417  0.0003   

   batch_size  dropout  
0          64      0.1  
1          64      0.5  
2          64      0.3  
3         128      0.5  
4         128      0.3  
5         256      0.3  

In [20]:
cifar_mlp_results_df = _tune_model_parallel('MLP', CIFAR_TRAIN_DATASET, CIFAR_VAL_DATASET, lrs, batch_sizes, dropouts, DEVICE)

print(cifar_mlp_results_df.head(10))

   best_train_loss  best_val_loss  best_train_acc  best_val_acc      lr  \
0         2.801137       2.806379        0.279150        0.2711  0.0003   
1         2.830690       2.831966        0.205325        0.2101  0.0001   
2         2.831149       2.829660        0.182225        0.2084  0.0003   
3         2.821721       2.821694        0.197425        0.1980  0.0003   
4         2.837129       2.835988        0.176350        0.1953  0.0003   
5         2.852122       2.854530        0.178375        0.1823  0.0010   
6         2.848612       2.841573        0.175300        0.1809  0.0003   
7         2.833938       2.834649        0.161075        0.1715  0.0003   
8         2.855559       2.858596        0.168725        0.1704  0.0010   
9         2.834563       2.834128        0.154925        0.1652  0.0003   

   batch_size  dropout  
0         256      0.1  
1          64      0.1  
2         128      0.1  
3         256      0.3  
4         128      0.5  
5         256      0.5  

In [21]:
# Tune the MNIST CNN model
mnist_cnn_results_df = _tune_model_parallel('CNN', MNIST_TRAIN_DATASET, MNIST_VAL_DATASET, lrs, batch_sizes, dropouts, DEVICE)

print(mnist_cnn_results_df.head(10))

KeyboardInterrupt: 

In [ ]:
cifar_cnn_results_df = _tune_model_parallel('CNN', CIFAR_TRAIN_DATASET, CIFAR_VAL_DATASET, lrs, batch_sizes, dropouts, DEVICE)

print(cifar_cnn_results_df.head(10))

#### Final parameters for the MLP model based on performance for both datasets:
* lr = 0.001 | 0.0003   Final choice: 0.001
* batch_size = 64 | 256     Final choice: 64
* dropout = 0.1

In [ ]:
TUNED_MLP_BATCH_SIZE = 64
TUNED_MLP_LR = 0.001
TUNED_MLP_DROPOUT = 0.1

# Train the CNN model on MNIST data using tuned parameters
tuned_mnist_mlp_model, tuned_mnist_mlp_history = training_loop('MLP', MNIST_TRAIN_DATASET, MNIST_VAL_DATASET, lr=TUNED_MLP_LR, batch_size=TUNED_MLP_BATCH_SIZE, dropout=TUNED_MLP_DROPOUT)

# Tuned history
tuned_mnist_mlp_history = {
    'epoch': tuned_mnist_mlp_history['epoch'],
    'train_loss': tuned_mnist_mlp_history['train_loss'],
    'val_loss': tuned_mnist_mlp_history['val_loss'],
    'train_acc': tuned_mnist_mlp_history['train_acc'],
    'val_acc': tuned_mnist_mlp_history['val_acc'],
    'lr': tuned_mnist_mlp_history['lr']
}

mnist_mlp_tuned_history_df = pd.DataFrame(tuned_mnist_mlp_history)
print(mnist_mlp_tuned_history_df)

In [ ]:
# Train the CNN model on CIFAR10 data using tuned parameters
tuned_cifar_mlp_model, tuned_cifar_mlp_history = training_loop('MLP', CIFAR_TRAIN_DATASET, CIFAR_VAL_DATASET, lr=TUNED_MLP_LR, batch_size=TUNED_MLP_BATCH_SIZE, dropout=TUNED_MLP_DROPOUT)

# Tuned history
tuned_cifar_mlp_history = {
    'epoch': tuned_cifar_mlp_history['epoch'],
    'train_loss': tuned_cifar_mlp_history['train_loss'],
    'val_loss': tuned_cifar_mlp_history['val_loss'],
    'train_acc': tuned_cifar_mlp_history['train_acc'],
    'val_acc': tuned_cifar_mlp_history['val_acc'],
    'lr': tuned_cifar_mlp_history['lr']
}

cifar_mlp_tuned_history_df = pd.DataFrame(tuned_cifar_mlp_history)
print(cifar_mlp_tuned_history_df)

#### Final CNN parameters based on performance for both datasets

* lr = 0.001
* batch_size = 256
* dropout = 0.1

In [ ]:
TUNED_MLP_BATCH_SIZE = 64
TUNED_MLP_LR = 0.001
TUNED_MLP_DROPOUT = 0.1

# Train the CNN model on MNIST data using tuned parameters
tuned_mnist_cnn_model, tuned_mnist_cnn_history = training_loop('CNN', MNIST_TRAIN_DATASET, MNIST_VAL_DATASET, lr=TUNED_MLP_LR, batch_size=TUNED_MLP_BATCH_SIZE, dropout=TUNED_MLP_DROPOUT)

# Tuned history
tuned_mnist_cnn_history = {
    'epoch': tuned_mnist_cnn_history['epoch'],
    'train_loss': tuned_mnist_cnn_history['train_loss'],
    'val_loss': tuned_mnist_cnn_history['val_loss'],
    'train_acc': tuned_mnist_cnn_history['train_acc'],
    'val_acc': tuned_mnist_cnn_history['val_acc'],
    'lr': tuned_mnist_cnn_history['lr']
}

mnist_cnn_tuned_history_df = pd.DataFrame(tuned_mnist_cnn_history)
print(mnist_cnn_tuned_history_df)

In [ ]:
# Train the CNN model on CIFAR10 data using tuned parameters
tuned_cifar_cnn_model, tuned_cifar_cnn_history = training_loop('CNN', CIFAR_TRAIN_DATASET, CIFAR_VAL_DATASET, lr=TUNED_MLP_LR, batch_size=TUNED_MLP_BATCH_SIZE, dropout=TUNED_MLP_DROPOUT)

# Tuned history
tuned_cifar_cnn_history = {
    'epoch': tuned_cifar_cnn_history['epoch'],
    'train_loss': tuned_cifar_cnn_history['train_loss'],
    'val_loss': tuned_cifar_cnn_history['val_loss'],
    'train_acc': tuned_cifar_cnn_history['train_acc'],
    'val_acc': tuned_cifar_cnn_history['val_acc'],
    'lr': tuned_cifar_cnn_history['lr']
}

cifar_cnn_tuned_history_df = pd.DataFrame(tuned_cifar_cnn_history)
print(cifar_cnn_tuned_history_df)